# Check Extractor — Fine-Tuning on AWS EC2
Fine-tunes **Qwen2.5-VL-7B-Instruct** on bank check images using Unsloth + QLoRA.

**Run cells top to bottom. Do not skip cells.**

## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

# Unsloth manages torch/transformers/bitsandbytes internally.
# Use colab-new extras — works on any CUDA machine, not just Colab.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-deps", "trl", "peft", "accelerate"
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "Pillow", "sentencepiece", "protobuf"
], check=True)

print("Done.")

## Cell 2 — Config
**Edit this cell** to match your EC2 setup before running anything else.

In [ ]:
from pathlib import Path

# ── Training data paths ───────────────────────────────────────────────
# Set these to wherever your training data lives on the EC2 instance.
# Default assumes data is in ~/training_data/ — change if different.
DATA_ROOT     = Path.home() / "training_data"
TRAIN_IMAGES  = DATA_ROOT / "images"
TRAIN_LABELS  = DATA_ROOT / "labels"
HOLDOUT_IMAGES = DATA_ROOT / "holdout" / "images"
HOLDOUT_LABELS = DATA_ROOT / "holdout" / "labels"

# ── Output ────────────────────────────────────────────────────────────
OUTPUT_DIR = Path.home() / "check_extractor_output"

# ── Model ─────────────────────────────────────────────────────────────
MODEL_NAME = "unsloth/Qwen2.5-VL-7B-Instruct"

# ── Hyperparameters ───────────────────────────────────────────────────
TRAIN_EPOCHS  = 3
LEARNING_RATE = 2e-4
GRAD_ACCUM    = 4
LORA_RANK     = 16
MAX_SEQ_LEN   = 2048
EVAL_SAMPLES  = 50  # holdout samples to evaluate (None = all)

print(f"Training images : {TRAIN_IMAGES}")
print(f"Training labels : {TRAIN_LABELS}")
print(f"Holdout images  : {HOLDOUT_IMAGES}")
print(f"Output dir      : {OUTPUT_DIR}")

## Cell 3 — Verify Environment

In [ ]:
import torch

assert torch.cuda.is_available(), "CUDA not available — need an NVIDIA GPU"
print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import unsloth
print(f"Unsloth: {unsloth.__version__}")

assert TRAIN_IMAGES.exists(),  f"Missing: {TRAIN_IMAGES}"
assert TRAIN_LABELS.exists(),  f"Missing: {TRAIN_LABELS}"

train_imgs   = len(list(TRAIN_IMAGES.glob("*.jpg")))  + len(list(TRAIN_IMAGES.glob("*.png")))
train_labels = len(list(TRAIN_LABELS.glob("*.json")))
print(f"Training : {train_imgs} images, {train_labels} labels")

holdout_imgs = 0
if HOLDOUT_IMAGES.exists():
    holdout_imgs   = len(list(HOLDOUT_IMAGES.glob("*.jpg"))) + len(list(HOLDOUT_IMAGES.glob("*.png")))
    holdout_labels = len(list(HOLDOUT_LABELS.glob("*.json")))
    print(f"Holdout  : {holdout_imgs} images, {holdout_labels} labels")
else:
    print("Holdout  : not found (evaluation will be skipped)")

assert train_imgs >= 50, f"Need at least 50 training images, found {train_imgs}"
print("\nPASSED")

## Cell 4 — Prompts and Dataset

In [ ]:
import json, re
from torch.utils.data import Dataset
from PIL import Image

SYSTEM_PROMPT = """You are a bank check data extraction model. Given an image of a check, extract the following fields and return ONLY a valid JSON object with no markdown, no code fences, no explanation. Use null (not empty string) for any field you cannot find or read.

If any text is partially covered, obscured, redacted, or not fully readable, set that field to null. Do NOT guess or reconstruct partially visible text.

Fields to extract:
- payorInstitution: The name of the bank printed on the check (e.g. \"BANK OF AMERICA\")
- payor: The account holder who wrote the check (name ONLY, no address)
- payee: The person or entity on the \"Pay to the order of\" line
- amount: Dollar amount as a plain number with two decimal places (e.g. \"2675.00\"). No dollar signs, commas, or words.
- account: Account number from the MICR line at the bottom of the check
- serial: Check serial/number (typically top-right or MICR line)
- checkDate: Date written on the check in YYYY-MM-DD format
- fractionalNumber: The fractional routing number near the check number, format: DD-DDD/DDDD (e.g. \"87-176/843\"). This is NOT the 9-digit routing number.
- calculatedRoutingNumber: Leave this as null — it will be computed in post-processing."""

USER_PROMPT = "Extract all fields from this bank check image and return a JSON object."

EXPECTED_FIELDS = [
    "payorInstitution", "payor", "payee", "amount",
    "account", "serial", "checkDate", "fractionalNumber",
]


def clean_label(label):
    label.pop("source_pool", None)
    label["calculatedRoutingNumber"] = None
    payor = label.get("payor")
    if payor and isinstance(payor, str):
        label["payor"] = payor.split("\n")[0].strip() or None
    return label


class CheckDataset(Dataset):
    def __init__(self, image_dir, label_dir):
        self.samples = []
        image_dir = Path(image_dir)
        label_dir = Path(label_dir)

        for img_path in sorted(list(image_dir.glob("*.jpg")) + list(image_dir.glob("*.png"))):
            label_path = label_dir / (img_path.stem + ".json")
            if not label_path.exists():
                continue
            with open(label_path) as f:
                label = clean_label(json.load(f))
            self.samples.append({
                "image_path": str(img_path),
                "ground_truth": json.dumps(label, indent=2),
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return {
            "messages": [
                {"role": "system",    "content": [{"type": "text",  "text": SYSTEM_PROMPT}]},
                {"role": "user",      "content": [{"type": "image", "image": s["image_path"]},
                                                  {"type": "text",  "text": USER_PROMPT}]},
                {"role": "assistant", "content": [{"type": "text",  "text": s["ground_truth"]}]},
            ]
        }


train_dataset = CheckDataset(TRAIN_IMAGES, TRAIN_LABELS)
print(f"Training samples loaded: {len(train_dataset)}")
sample = train_dataset[0]
print(f"Sample image: {sample['messages'][1]['content'][0]['image']}")
print(f"Sample label (first 200 chars): {sample['messages'][2]['content'][0]['text'][:200]}")

## Cell 5 — Load Model

In [ ]:
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

print(f"Model loaded: {MODEL_NAME}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## Cell 6 — Apply LoRA Adapters

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    random_state=42,
    use_rslora=False,
)

model.print_trainable_parameters()

## Cell 7 — Baseline Evaluation (before training)

In [ ]:
import os

def parse_json_output(text):
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}", cleaned, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
    return None


def run_inference(image_path, model, tokenizer):
    image = Image.open(image_path).convert("RGB")
    messages = [
        {"role": "system", "content": [{"type": "text",  "text": SYSTEM_PROMPT}]},
        {"role": "user",   "content": [{"type": "image", "image": image},
                                       {"type": "text",  "text": USER_PROMPT}]},
    ]
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text=[text_input], images=[image], return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=512, temperature=0.1,
            do_sample=False, repetition_penalty=1.1,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def evaluate(model, tokenizer, image_dir, label_dir, max_samples=None, label="Evaluation"):
    FastVisionModel.for_inference(model)
    image_dir = Path(image_dir)
    label_dir = Path(label_dir)

    image_files = sorted(list(image_dir.glob("*.jpg")) + list(image_dir.glob("*.png")))
    if max_samples:
        image_files = image_files[:max_samples]

    field_correct = {f: 0 for f in EXPECTED_FIELDS}
    field_total   = {f: 0 for f in EXPECTED_FIELDS}
    results = []

    for i, img_path in enumerate(image_files):
        label_path = label_dir / (img_path.stem + ".json")
        if not label_path.exists():
            continue
        with open(label_path) as f:
            truth = clean_label(json.load(f))

        raw = run_inference(str(img_path), model, tokenizer)
        predicted = parse_json_output(raw) or {}

        sample_result = {"image": img_path.name, "predicted": predicted, "truth": truth, "fields": {}}
        for field in EXPECTED_FIELDS:
            t = truth.get(field)
            p = predicted.get(field)
            t_str = str(t).strip().lower() if t else ""
            p_str = str(p).strip().lower() if p else ""
            if not t_str:
                continue
            field_total[field] += 1
            if t_str == p_str:
                field_correct[field] += 1
                sample_result["fields"][field] = "correct"
            else:
                sample_result["fields"][field] = f"mismatch (gt={t!r}, pred={p!r})"
        results.append(sample_result)

        if (i + 1) % 10 == 0:
            print(f"  {label}: {i + 1}/{len(image_files)} evaluated...")

    total_c  = sum(field_correct.values())
    total_t  = sum(field_total.values())
    accuracy = (total_c / total_t * 100) if total_t > 0 else 0

    print(f"\n{label} — Per-field accuracy:")
    for field in EXPECTED_FIELDS:
        c, t = field_correct[field], field_total[field]
        pct  = (c / t * 100) if t > 0 else 0
        print(f"  {field:30s}: {c:4d}/{t:4d} ({pct:5.1f}%)")
    print(f"\n  Overall: {total_c}/{total_t} ({accuracy:.1f}%)")

    return {"accuracy": accuracy, "field_correct": field_correct, "field_total": field_total, "details": results}


os.makedirs(OUTPUT_DIR, exist_ok=True)

if HOLDOUT_IMAGES.exists():
    baseline = evaluate(model, tokenizer, HOLDOUT_IMAGES, HOLDOUT_LABELS,
                        max_samples=EVAL_SAMPLES, label="BASELINE")
    with open(OUTPUT_DIR / "baseline_eval.json", "w") as f:
        json.dump(baseline, f, indent=2)
    print(f"Saved: {OUTPUT_DIR}/baseline_eval.json")
else:
    print("No holdout set found — skipping baseline evaluation")
    baseline = {"accuracy": 0}

## Cell 8 — Train

In [ ]:
import time
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=10,
    learning_rate=LEARNING_RATE,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25,
    save_strategy="epoch",
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    remove_unused_columns=False,
    report_to="none",
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
    max_seq_length=MAX_SEQ_LEN,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
)

total_steps = len(train_dataset) * TRAIN_EPOCHS // GRAD_ACCUM
print(f"Samples        : {len(train_dataset)}")
print(f"Epochs         : {TRAIN_EPOCHS}")
print(f"Optimizer steps: ~{total_steps}")
print(f"Starting training...\n")

start = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start

print(f"\nTraining complete.")
print(f"Runtime   : {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

## Cell 9 — Post-Training Evaluation

In [ ]:
if HOLDOUT_IMAGES.exists():
    post_eval = evaluate(model, tokenizer, HOLDOUT_IMAGES, HOLDOUT_LABELS,
                         max_samples=EVAL_SAMPLES, label="POST-TRAINING")
    with open(OUTPUT_DIR / "post_training_eval.json", "w") as f:
        json.dump(post_eval, f, indent=2)

    delta = post_eval["accuracy"] - baseline["accuracy"]
    print(f"\nBaseline       : {baseline['accuracy']:.1f}%")
    print(f"Post-training  : {post_eval['accuracy']:.1f}%")
    print(f"Delta          : {delta:+.1f}%")
    print(f"Saved: {OUTPUT_DIR}/post_training_eval.json")
else:
    print("No holdout set — skipping post-training evaluation")

## Cell 10 — Save LoRA Adapter

In [ ]:
adapter_dir = OUTPUT_DIR / "adapter"
os.makedirs(adapter_dir, exist_ok=True)
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"Adapter saved: {adapter_dir}")

## Cell 11 — Export to GGUF (Q4_K_M)
This takes 5–15 minutes. Do not interrupt.

In [ ]:
import subprocess, sys

gguf_dir = OUTPUT_DIR / "gguf"
os.makedirs(gguf_dir, exist_ok=True)

print("Merging adapter and exporting to GGUF (q4_k_m)...")

model.save_pretrained_gguf(
    str(gguf_dir),
    tokenizer,
    quantization_method="q4_k_m",
)

# Unsloth sometimes saves safetensors instead of GGUF for vision models.
# If that happens, fall back to llama.cpp.
gguf_files = [f for f in os.listdir(gguf_dir) if f.endswith(".gguf")]

if not gguf_files:
    print("Unsloth produced safetensors — using llama.cpp fallback")

    llama_cpp = OUTPUT_DIR / "llama.cpp"
    if not llama_cpp.exists():
        print("Cloning llama.cpp...")
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/ggml-org/llama.cpp", str(llama_cpp)], check=True)

    print("Installing llama.cpp Python requirements...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(llama_cpp / "requirements.txt")], check=True)

    f16_path = gguf_dir / "check-extractor-f16.gguf"
    print("Converting to f16 GGUF...")
    subprocess.run([
        sys.executable, str(llama_cpp / "convert_hf_to_gguf.py"),
        str(gguf_dir), "--outfile", str(f16_path), "--outtype", "f16",
    ], check=True)

    print("Building llama-quantize...")
    subprocess.run(["cmake", "-B", "build"], cwd=str(llama_cpp), check=True, capture_output=True)
    subprocess.run(["cmake", "--build", "build", "--target", "llama-quantize", "-j4"],
                   cwd=str(llama_cpp), check=True, capture_output=True)

    q4_path      = gguf_dir / "check-extractor-q4km.gguf"
    quantize_bin = llama_cpp / "build" / "bin" / "llama-quantize"
    print("Quantizing to Q4_K_M...")
    subprocess.run([str(quantize_bin), str(f16_path), str(q4_path), "Q4_K_M"], check=True)

    if q4_path.exists():
        os.remove(f16_path)
        for sf in gguf_dir.glob("*.safetensors"):
            os.remove(sf)
        print("Cleaned up f16 and safetensors.")

    gguf_files = [f for f in os.listdir(gguf_dir) if f.endswith(".gguf")]

# Write Modelfile
modelfile_path = gguf_dir / "Modelfile"
if not modelfile_path.exists() and gguf_files:
    with open(modelfile_path, "w") as mf:
        mf.write(f"FROM ./{gguf_files[0]}\n")
        mf.write("PARAMETER temperature 0.1\n")
        mf.write(f'SYSTEM """{SYSTEM_PROMPT}"""\n')
    print("Modelfile written.")

print("\nGGUF export complete. Files:")
for f in os.listdir(gguf_dir):
    fp = gguf_dir / f
    if fp.is_file():
        print(f"  {f}  ({os.path.getsize(fp) / 1e6:.1f} MB)")

## Cell 12 — Done
Copy `output/gguf/` off the EC2 instance, then on your Mac:

```bash
# requires Ollama >= 0.7.0
cd output/gguf
ollama create check-extractor -f Modelfile
ollama list
```

Then update `ollama.py` to use `model="check-extractor"`.

In [ ]:
print("="*60)
print("COMPLETE")
print(f"Output directory : {OUTPUT_DIR}")
print(f"GGUF directory   : {OUTPUT_DIR / 'gguf'}")
print(f"Adapter backup   : {OUTPUT_DIR / 'adapter'}")
print("="*60)